# Importing necessary libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import warnings 
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("Players stats.csv")
df

,Player,Span,Mat,Inns,NO,Runs,HS,Ave,BF,SR,100,50,0s
0,SR Tendulkar (IND),1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96,20
1,V Kohli (IND),2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76,18
2,KC Sangakkara (Asia/ICC/SL),2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93,15
3,RT Ponting (AUS/ICC),1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82,20
4,ST Jayasuriya (Asia/SL),1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,PR Reiffel (AUS),1992-1999,92,57,21,503,58,13.97,709,70.94,0,1,5
744,AR Nurse (WI),2016-2019,54,40,14,502,44,19.30,513,97.85,0,0,2
745,JE Emburey (ENG),1980-1993,61,45,10,501,34,14.31,664,75.45,0,0,3
746,Ghulam Shabber (UAE),2017-2019,23,23,0,500,90,21.73,800,62.50,0,4,2


In [3]:
df.set_index("Player")

,Span,Mat,Inns,NO,Runs,HS,Ave,BF,SR,100,50,0s
Player,,,,,,,,,,,,
SR Tendulkar (IND),1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96,20
V Kohli (IND),2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76,18
KC Sangakkara (Asia/ICC/SL),2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93,15
RT Ponting (AUS/ICC),1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82,20
ST Jayasuriya (Asia/SL),1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68,34
...,...,...,...,...,...,...,...,...,...,...,...,...
PR Reiffel (AUS),1992-1999,92,57,21,503,58,13.97,709,70.94,0,1,5
AR Nurse (WI),2016-2019,54,40,14,502,44,19.30,513,97.85,0,0,2
JE Emburey (ENG),1980-1993,61,45,10,501,34,14.31,664,75.45,0,0,3


In [4]:
df.shape

(748, 13)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  748 non-null    object 
 1   Span    748 non-null    object 
 2   Mat     748 non-null    int64  
 3   Inns    748 non-null    int64  
 4   NO      748 non-null    int64  
 5   Runs    748 non-null    int64  
 6   HS      748 non-null    object 
 7   Ave     748 non-null    float64
 8   BF      748 non-null    int64  
 9   SR      748 non-null    float64
 10  100     748 non-null    int64  
 11  50      748 non-null    int64  
 12  0s      748 non-null    int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 76.1+ KB


# Overall missing value percentage in data

In [6]:
total_cells = df.size
total_missing = df.isnull().sum().sum()

overall_missing_percent = (total_missing / total_cells) * 100
print(f"Overall Missing Values: {overall_missing_percent:.2f}%")

Overall Missing Values: 0.00%


In [7]:
df.drop(columns = "0s",inplace = True)

In [8]:
df.columns

Index(['Player', 'Span', 'Mat', 'Inns', 'NO', 'Runs', 'HS', 'Ave', 'BF', 'SR',
       '100', '50'],
      dtype='object')

# Renaming the column for better understanding

In [9]:
df = df.rename(columns={
    "Player": "Player_Name",
    "Mat": "Matches",
    "Inns": "Innings",
    "Runs": "Total_Runs",
    "Ave": "Batting_Average",
    "SR": "Strike_Rate",
    "NO":"Not_Out",
    "HS":"Highest_Score",
    "BF":"Ball_Faced",
    "100":"100's",
    "50":"50's"
})
df.head()


,Player_Name,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar (IND),1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96
1,V Kohli (IND),2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara (Asia/ICC/SL),2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting (AUS/ICC),1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya (Asia/SL),1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68


# Adding column country to get country names

In [10]:
df[['Player_Names','Country']] = df['Player_Name'].str.rsplit(')',n=1,expand = True)

# Replacing parenthesis of country names

In [11]:
df[['Player_Names','Country']] = df['Player_Name'].str.rsplit('(',n=1,expand = True)

In [12]:
df["Country"] = df["Country"].str.replace(")","")

In [13]:
df.drop(columns = "Player_Name", inplace = True)

In [14]:
cols = ['Player_Names', 'Country'] + [c for c in df.columns if c not in ['Player_Names', 'Country']]
df = df[cols]
df.head()

,Player_Names,Country,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,IND,1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96
1,V Kohli,IND,2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Asia/ICC/SL,2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,AUS/ICC,1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Asia/SL,1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68


# Checking Duplicates

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df.isna().sum()

Player_Names       0
Country            0
Span               0
Matches            0
Innings            0
Not_Out            0
Total_Runs         0
Highest_Score      0
Batting_Average    0
Ball_Faced         0
Strike_Rate        0
100's              0
50's               0
dtype: int64

In [17]:
df.shape

(748, 13)

In [18]:
df.head(10)

,Player_Names,Country,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,IND,1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96
1,V Kohli,IND,2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Asia/ICC/SL,2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,AUS/ICC,1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Asia/SL,1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68
5,DPMD Jayawardene,Asia/SL,1998-2015,448,418,39,12650,144,33.37,16020,78.96,19,77
6,Inzamam-ul-Haq,Asia/PAK,1991-2007,378,350,53,11739,137*,39.52,15812,74.24,10,83
7,JH Kallis,Afr/ICC/SA,1996-2014,328,314,53,11579,139,44.36,15885,72.89,17,86
8,RG Sharma,IND,2007-2025,279,271,37,11516,264,49.21,12402,92.85,33,61
9,SC Ganguly,Asia/IND,1992-2007,311,300,23,11363,183,41.02,15416,73.70,22,72


In [19]:
df["Country"].unique()

array(['IND', 'Asia/ICC/SL', 'AUS/ICC', 'Asia/SL', 'Asia/PAK',
       'Afr/ICC/SA', 'Asia/IND', 'Asia/ICC/IND', 'ICC/WI', 'SL', 'Afr/SA',
       'PAK', 'WI', 'NZ', 'BAN', 'SA', 'Asia/ICC/PAK', 'ICC/NZ', 'AUS',
       'ENG/IRE', 'ENG', 'ZIM', 'IRE', 'ENG/ICC', 'Afr/ZIM', 'AFG',
       'SCOT', 'Asia/BAN', 'Afr/KENYA', 'AUS/SA', 'USA', 'NAM', 'NED',
       'KENYA', 'PNG', 'NEP', 'CAN', 'UAE', 'OMA', 'AUS/NZ', 'HKG/NZ',
       'ENG/PNG', 'HKG', 'BER', 'USA/WI'], dtype=object)

In [20]:
df["Country"] = df["Country"].str.replace(r"(Asia|Afr|ICC)/", "", regex=True)
c_names = {
    "IND": "India",
    "AUS": "Australia",
    "SL": "Sri Lanka",
    "PAK": "Pakistan",
    "SA": "South Africa",
    "WI": "West Indies",
    "NZ": "New Zealand",
    "BAN": "Bangladesh",
    "ENG": "England",
    "IRE": "Ireland",
    "ZIM": "Zimbabwe",
    "AFG": "Afghanistan",
    "SCOT": "Scotland",
    "KENYA": "Kenya",
    "USA": "United States",
    "NAM": "Namibia",
    "NED": "Netherlands",
    "PNG": "Papua New Guinea",
    "CAN": "Canada",
    "UAE": "United Arab Emirates",
    "NEP": "Nepal",
    "OMA": "Oman",
    "AUS/ICC":"Australia",
    "ENG/ICC":"England",
    "AUS/SA":"Australia/South Africa",
    "AUS/NZ":"Australia/New Zealand",
    'ENG/IRE':'England/Ireland'
}
df["Country"] = df["Country"].replace(c_names)

In [21]:
df.head(10)

,Player_Names,Country,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96
1,V Kohli,India,2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68
5,DPMD Jayawardene,Sri Lanka,1998-2015,448,418,39,12650,144,33.37,16020,78.96,19,77
6,Inzamam-ul-Haq,Pakistan,1991-2007,378,350,53,11739,137*,39.52,15812,74.24,10,83
7,JH Kallis,South Africa,1996-2014,328,314,53,11579,139,44.36,15885,72.89,17,86
8,RG Sharma,India,2007-2025,279,271,37,11516,264,49.21,12402,92.85,33,61
9,SC Ganguly,India,1992-2007,311,300,23,11363,183,41.02,15416,73.70,22,72


In [22]:
df.head(10)

,Player_Names,Country,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96
1,V Kohli,India,2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68
5,DPMD Jayawardene,Sri Lanka,1998-2015,448,418,39,12650,144,33.37,16020,78.96,19,77
6,Inzamam-ul-Haq,Pakistan,1991-2007,378,350,53,11739,137*,39.52,15812,74.24,10,83
7,JH Kallis,South Africa,1996-2014,328,314,53,11579,139,44.36,15885,72.89,17,86
8,RG Sharma,India,2007-2025,279,271,37,11516,264,49.21,12402,92.85,33,61
9,SC Ganguly,India,1992-2007,311,300,23,11363,183,41.02,15416,73.70,22,72


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player_Names     748 non-null    object 
 1   Country          748 non-null    object 
 2   Span             748 non-null    object 
 3   Matches          748 non-null    int64  
 4   Innings          748 non-null    int64  
 5   Not_Out          748 non-null    int64  
 6   Total_Runs       748 non-null    int64  
 7   Highest_Score    748 non-null    object 
 8   Batting_Average  748 non-null    float64
 9   Ball_Faced       748 non-null    int64  
 10  Strike_Rate      748 non-null    float64
 11  100's            748 non-null    int64  
 12  50's             748 non-null    int64  
dtypes: float64(2), int64(7), object(4)
memory usage: 76.1+ KB


# Covreting span into two columns 

In [24]:
df[['Start_Year', 'End_Year']] = df['Span'].str.split('-', expand=True)

In [25]:
df

,Player_Names,Country,Span,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's,Start_Year,End_Year
0,SR Tendulkar,India,1989-2012,463,452,41,18426,200*,44.83,21368,86.23,49,96,1989,2012
1,V Kohli,India,2008-2025,308,296,47,14557,183,58.46,15543,93.65,53,76,2008,2025
2,KC Sangakkara,Sri Lanka,2000-2015,404,380,41,14234,169,41.98,18048,78.86,25,93,2000,2015
3,RT Ponting,Australia,1995-2012,375,365,39,13704,164,42.03,17046,80.39,30,82,1995,2012
4,ST Jayasuriya,Sri Lanka,1989-2011,445,433,18,13430,189,32.36,14725,91.20,28,68,1989,2011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,PR Reiffel,Australia,1992-1999,92,57,21,503,58,13.97,709,70.94,0,1,1992,1999
744,AR Nurse,West Indies,2016-2019,54,40,14,502,44,19.30,513,97.85,0,0,2016,2019
745,JE Emburey,England,1980-1993,61,45,10,501,34,14.31,664,75.45,0,0,1980,1993
746,Ghulam Shabber,United Arab Emirates,2017-2019,23,23,0,500,90,21.73,800,62.50,0,4,2017,2019


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player_Names     748 non-null    object 
 1   Country          748 non-null    object 
 2   Span             748 non-null    object 
 3   Matches          748 non-null    int64  
 4   Innings          748 non-null    int64  
 5   Not_Out          748 non-null    int64  
 6   Total_Runs       748 non-null    int64  
 7   Highest_Score    748 non-null    object 
 8   Batting_Average  748 non-null    float64
 9   Ball_Faced       748 non-null    int64  
 10  Strike_Rate      748 non-null    float64
 11  100's            748 non-null    int64  
 12  50's             748 non-null    int64  
 13  Start_Year       748 non-null    object 
 14  End_Year         748 non-null    object 
dtypes: float64(2), int64(7), object(6)
memory usage: 87.8+ KB


# Replacing *

In [27]:
df['Highest_Score'] = df['Highest_Score'].astype(str).str.replace('*', '', regex=False)
df['Highest_Score'] = pd.to_numeric(df['Highest_Score'])

# Making new order

In [28]:
new_order = [
    'Player_Names', 'Country', 'Start_Year', 'End_Year', 
    'Matches', 'Innings', 'Not_Out', 'Total_Runs', 
    'Highest_Score', 'Batting_Average', 'Ball_Faced', 
    'Strike_Rate', "100's", "50's"
]

df = df[[col for col in new_order if col in df.columns]]

In [29]:
df.head()

,Player_Names,Country,Start_Year,End_Year,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989,2012,463,452,41,18426,200,44.83,21368,86.23,49,96
1,V Kohli,India,2008,2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000,2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995,2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989,2011,445,433,18,13430,189,32.36,14725,91.20,28,68


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player_Names     748 non-null    object 
 1   Country          748 non-null    object 
 2   Start_Year       748 non-null    object 
 3   End_Year         748 non-null    object 
 4   Matches          748 non-null    int64  
 5   Innings          748 non-null    int64  
 6   Not_Out          748 non-null    int64  
 7   Total_Runs       748 non-null    int64  
 8   Highest_Score    748 non-null    int64  
 9   Batting_Average  748 non-null    float64
 10  Ball_Faced       748 non-null    int64  
 11  Strike_Rate      748 non-null    float64
 12  100's            748 non-null    int64  
 13  50's             748 non-null    int64  
dtypes: float64(2), int64(8), object(4)
memory usage: 81.9+ KB


In [31]:
df['Player_Names'].str.upper()

0        SR TENDULKAR 
1             V KOHLI 
2       KC SANGAKKARA 
3          RT PONTING 
4       ST JAYASURIYA 
            ...       
743        PR REIFFEL 
744          AR NURSE 
745        JE EMBUREY 
746    GHULAM SHABBER 
747          GP SWANN 
Name: Player_Names, Length: 748, dtype: object

In [32]:
df[["Start_Year","End_Year"]] = df[["Start_Year","End_Year"]].astype(object)

In [33]:
df["Player_Names"].nunique()

747

In [34]:
df

,Player_Names,Country,Start_Year,End_Year,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989,2012,463,452,41,18426,200,44.83,21368,86.23,49,96
1,V Kohli,India,2008,2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000,2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995,2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989,2011,445,433,18,13430,189,32.36,14725,91.20,28,68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,PR Reiffel,Australia,1992,1999,92,57,21,503,58,13.97,709,70.94,0,1
744,AR Nurse,West Indies,2016,2019,54,40,14,502,44,19.30,513,97.85,0,0
745,JE Emburey,England,1980,1993,61,45,10,501,34,14.31,664,75.45,0,0
746,Ghulam Shabber,United Arab Emirates,2017,2019,23,23,0,500,90,21.73,800,62.50,0,4


In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 748 entries, 0 to 747
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player_Names     748 non-null    object 
 1   Country          748 non-null    object 
 2   Start_Year       748 non-null    object 
 3   End_Year         748 non-null    object 
 4   Matches          748 non-null    int64  
 5   Innings          748 non-null    int64  
 6   Not_Out          748 non-null    int64  
 7   Total_Runs       748 non-null    int64  
 8   Highest_Score    748 non-null    int64  
 9   Batting_Average  748 non-null    float64
 10  Ball_Faced       748 non-null    int64  
 11  Strike_Rate      748 non-null    float64
 12  100's            748 non-null    int64  
 13  50's             748 non-null    int64  
dtypes: float64(2), int64(8), object(4)
memory usage: 81.9+ KB


# Some rows removing for better visiualization

In [36]:
df['Player_Names'].unique()

array(['SR Tendulkar ', 'V Kohli ', 'KC Sangakkara ', 'RT Ponting ',
       'ST Jayasuriya ', 'DPMD Jayawardene ', 'Inzamam-ul-Haq ',
       'JH Kallis ', 'RG Sharma ', 'SC Ganguly ', 'R Dravid ',
       'MS Dhoni ', 'CH Gayle ', 'BC Lara ', 'TM Dilshan ',
       'Mohammad Yousuf ', 'AC Gilchrist ', 'AB de Villiers ',
       'M Azharuddin ', 'PA de Silva ', 'Saeed Anwar ', 'S Chanderpaul ',
       'Yuvraj Singh ', 'DL Haynes ', 'LRPL Taylor ', 'MS Atapattu ',
       'Tamim Iqbal ', 'V Sehwag ', 'HM Amla ', 'HH Gibbs ',
       'Shahid Afridi ', 'SP Fleming ', 'MJ Clarke ', 'Mushfiqur Rahim ',
       'EJG Morgan ', 'Shakib Al Hasan ', 'SR Waugh ', 'Shoaib Malik ',
       'A Ranatunga ', 'MJ Guptill ', 'JE Root ', 'KS Williamson ',
       'Younis Khan ', 'Saleem Malik ', 'Q de Kock ', 'NJ Astle ',
       'GC Smith ', 'WU Tharanga ', 'DA Warner ', 'MG Bevan ',
       'G Kirsten ', 'S Dhawan ', 'A Flower ', 'IVA Richards ',
       'BRM Taylor ', 'Mohammad Hafeez ', 'GW Flower ', 'Ijaz Ahmed

In [37]:
df = df[df["Country"] != "England/Ireland"]
df = df[df["Country"] != "Australia/South Africa"]
df = df[df["Country"] != "Australia/New Zealand"]
df

,Player_Names,Country,Start_Year,End_Year,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989,2012,463,452,41,18426,200,44.83,21368,86.23,49,96
1,V Kohli,India,2008,2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000,2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995,2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989,2011,445,433,18,13430,189,32.36,14725,91.20,28,68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,PR Reiffel,Australia,1992,1999,92,57,21,503,58,13.97,709,70.94,0,1
744,AR Nurse,West Indies,2016,2019,54,40,14,502,44,19.30,513,97.85,0,0
745,JE Emburey,England,1980,1993,61,45,10,501,34,14.31,664,75.45,0,0
746,Ghulam Shabber,United Arab Emirates,2017,2019,23,23,0,500,90,21.73,800,62.50,0,4


In [38]:
df['Country'].unique()

array(['India', 'Sri Lanka', 'Australia', 'Pakistan', 'South Africa',
       'West Indies', 'New Zealand', 'Bangladesh', 'England', 'Zimbabwe',
       'Ireland', 'Afghanistan', 'Scotland', 'Kenya', 'United States',
       'Namibia', 'Netherlands', 'Papua New Guinea', 'Nepal', 'Canada',
       'United Arab Emirates', 'Oman', 'HKG/NZ', 'ENG/PNG', 'HKG', 'BER',
       'USA/WI'], dtype=object)

In [39]:
df.drop(df[df['Player_Names'] == 'SC Williams'].index, inplace=True)

# Converting DataFrame into CSV file

In [40]:
df.to_csv("Player_stats_Clean_Dataset.csv",index = False)

In [41]:
df


,Player_Names,Country,Start_Year,End_Year,Matches,Innings,Not_Out,Total_Runs,Highest_Score,Batting_Average,Ball_Faced,Strike_Rate,100's,50's
0,SR Tendulkar,India,1989,2012,463,452,41,18426,200,44.83,21368,86.23,49,96
1,V Kohli,India,2008,2025,308,296,47,14557,183,58.46,15543,93.65,53,76
2,KC Sangakkara,Sri Lanka,2000,2015,404,380,41,14234,169,41.98,18048,78.86,25,93
3,RT Ponting,Australia,1995,2012,375,365,39,13704,164,42.03,17046,80.39,30,82
4,ST Jayasuriya,Sri Lanka,1989,2011,445,433,18,13430,189,32.36,14725,91.20,28,68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,PR Reiffel,Australia,1992,1999,92,57,21,503,58,13.97,709,70.94,0,1
744,AR Nurse,West Indies,2016,2019,54,40,14,502,44,19.30,513,97.85,0,0
745,JE Emburey,England,1980,1993,61,45,10,501,34,14.31,664,75.45,0,0
746,Ghulam Shabber,United Arab Emirates,2017,2019,23,23,0,500,90,21.73,800,62.50,0,4
